# RAG on Azure — Day 2: The Real Pipeline 

**Use case:** Employee Handbook Q&A, done properly.

On Day 1 we stuffed a whole document into the prompt. That doesn't scale — real corpora are too big for the context window. Today we build the actual RAG pipeline:

**chunk → embed → index in Azure AI Search → retrieve top-k → generate (with citations)**


## Prerequisites (recap from Day 1, plus Search)

You need the same Azure OpenAI deployments as Day 1, **and** an Azure AI Search service. Update your `.env`:

```dotenv
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_API_VERSION=2024-10-21
AZURE_OPENAI_CHAT_DEPLOYMENT=gpt-4o
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large

AZURE_SEARCH_ENDPOINT=https://<your-search>.search.windows.net
AZURE_SEARCH_API_KEY=<your-search-admin-key>
AZURE_SEARCH_INDEX=rag-handbook-day2
```

> The Search **admin** key is required because we create an index and upload documents (a query key is read-only).

## 0. Install dependencies

**Restart the kernel after installing.**

In [1]:
%pip install -q "openai>=1.55.3" "azure-search-documents>=11.5.1" pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# 1) Locate the .env file: try the explicit path, then auto-discovery.
explicit = Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env")
env_path = explicit if explicit.exists() else (Path(find_dotenv()) if find_dotenv() else None)

print("Working dir   :", Path.cwd())
print("Explicit path :", explicit, "->", "EXISTS" if explicit.exists() else "NOT FOUND")
print("Using .env at :", env_path if env_path else "NONE FOUND")

# 2) Parse the file ourselves with utf-8-sig (strips a BOM that Notepad often adds)
#    and tolerate stray quotes/spaces. This is more robust than load_dotenv alone.
if env_path and Path(env_path).exists():
    keys_found = []
    for line in Path(env_path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        os.environ[k] = v.strip().strip('"').strip("'")
        keys_found.append(k)
    print("Keys found in file:", keys_found)
    load_dotenv(env_path, override=False)  # secondary, harmless
else:
    raise FileNotFoundError(
        "No .env file found. Make sure it is literally named '.env' (not '.env.txt'), "
        "and either sits next to this notebook or update the explicit path above."
    )

# 3) Read values
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT          = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o")
EMBEDDING_DEPLOYMENT     = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_SEARCH_ENDPOINT    = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY     = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME               = os.getenv("AZURE_SEARCH_INDEX", "rag-handbook-day2")

# 4) Masked confirmation (never print full secrets)
def _mask(v):
    return f"{v[:4]}...{v[-4:]} (len {len(v)})" if v else "MISSING"
#print("AZURE_OPENAI_API_KEY :", _mask(AZURE_OPENAI_API_KEY))
#print("AZURE_SEARCH_API_KEY :", _mask(AZURE_SEARCH_API_KEY))

# 5) Clear, actionable error if anything is still missing
missing = [n for n, v in [
    ("AZURE_OPENAI_ENDPOINT", AZURE_OPENAI_ENDPOINT),
    ("AZURE_OPENAI_API_KEY",  AZURE_OPENAI_API_KEY),
    ("AZURE_SEARCH_ENDPOINT", AZURE_SEARCH_ENDPOINT),
    ("AZURE_SEARCH_API_KEY",  AZURE_SEARCH_API_KEY),
] if not v]

if missing:
    raise ValueError(
        "Missing from .env: " + ", ".join(missing) +
        f"\n.env used: {env_path}"
        "\nCompare the 'Keys found in file' list above against these names. "
        "If a name is absent or misspelled there, fix the line in your .env "
        "(format: KEY=value, no spaces around '='), then RESTART THE KERNEL and re-run."
    )

print("\nConfig loaded OK. Index name:", INDEX_NAME)


Working dir   : c:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\notebooks
Explicit path : C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env -> EXISTS
Using .env at : C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env
Keys found in file: ['AZURE_DOC_INTEL_ENDPOINT', 'AZURE_DOC_INTEL_KEY', 'AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'AZURE_OPENAI_API_VERSION', 'AZURE_OPENAI_CHAT_DEPLOYMENT', 'AZURE_SEARCH_ENDPOINT', 'AZURE_SEARCH_API_KEY', 'AZURE_SEARCH_INDEX', 'AZURE_STORAGE_ACCOUNT_NAME', 'AZURE_STORAGE_CONNECTION_STRING', 'AZURE_STORAGE_ACCOUNT_NAME', 'AZURE_STORAGE_CONTAINER_NAME']
AZURE_OPENAI_API_KEY : 1lqh...iOl6 (len 84)
AZURE_SEARCH_API_KEY : DX8z...zwj4 (len 52)

Config loaded OK. Index name: ragpipeline-index


## 2. Create the clients

One Azure OpenAI client (chat + embeddings) and the Azure AI Search clients (one for managing the index, one for querying/uploading).

In [3]:
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

aoai = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

search_cred = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=search_cred)

def get_search_client():
    return SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_cred)

print("Clients ready.")

Clients ready.


## 3. Load documents

Reads every PDF in `./data`. If none are found, it falls back to built-in sample text so the notebook runs out of the box. The print shows the **actual resolved path** it checked, so there's no guessing where it looked.

In [13]:
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path("../data")   # use Path("..") / "data" if your PDFs live one folder up

SAMPLE_TEXT = """Acme Corp Employee Handbook (excerpt)

Paid Time Off (PTO)
Full-time employees accrue 15 days of paid time off (PTO) per year.
New hires begin accruing PTO after completing a 90-day probation period.
Unused PTO of up to 5 days may be carried over into the following year.

Remote Work
Remote work is permitted up to 3 days per week with manager approval.
Employees must be reachable during core hours of 10am to 3pm local time.

Expenses
Business expenses are reimbursed within 30 days of an approved submission.
Receipts are required for any expense over 25 dollars.
"""

def load_documents():
    print("Looking in:", DATA_DIR.resolve())
    docs = []
    pdfs = sorted(DATA_DIR.glob("*.pdf")) if DATA_DIR.exists() else []
    if pdfs:
        for pdf in pdfs:
            reader = PdfReader(str(pdf))
            text = "\n".join((page.extract_text() or "") for page in reader.pages)
            docs.append((pdf.name, text))
            print(f"  Loaded {pdf.name}: {len(reader.pages)} pages, {len(text)} chars")
    else:
        print("  No PDFs found — using built-in sample text.")
        docs.append(("sample_handbook.txt", SAMPLE_TEXT))
    return docs

documents = load_documents()
print(f"\n{len(documents)} document(s) loaded.")

Looking in: C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\data
  Loaded 01_PTO_and_Leave_Policy.pdf: 1 pages, 1603 chars
  Loaded 02_New_Hire_Onboarding_Guide.pdf: 1 pages, 1268 chars
  Loaded 03_Remote_Work_Policy.pdf: 1 pages, 1377 chars
  Loaded 04_Employee_Benefits_Overview.pdf: 1 pages, 1040 chars
  Loaded 05_Code_of_Conduct.pdf: 1 pages, 947 chars

5 document(s) loaded.


## 4. Chunk the text

Why chunk? Embeddings capture meaning best over focused passages, and we only want to send the LLM the relevant pieces — not whole documents. We use ~1000-character chunks with 150-character **overlap** so a sentence split across a boundary still appears intact in one chunk.


In [14]:
def chunk_text(text, chunk_size=1000, overlap=150):
    """Split text into overlapping chunks, preferring clean breaks on newlines/sentences."""
    text = text.strip()
    if not text:
        return []
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            window = text[start:end]
            # prefer to break on a paragraph, then sentence, then word boundary
            break_at = max(window.rfind("\n"), window.rfind(". "), window.rfind(" "))
            if break_at > chunk_size * 0.5:
                end = start + break_at + 1
        chunks.append(text[start:end].strip())
        start = max(0, end - overlap)  # step back by `overlap` to keep context
    return [c for c in chunks if c]

# Build flat chunk records with metadata.
# NOTE: Azure AI Search keys allow only letters, digits, _ , - , = (no spaces or dots).
records = []
for source, text in documents:
    safe_source = source.replace(" ", "_").replace(".", "_")
    for i, chunk in enumerate(chunk_text(text)):
        records.append({"id": f"{safe_source}-{i}", "content": chunk, "source": source})

print(f"Created {len(records)} chunks from {len(documents)} document(s).")
print("\nExample chunk:\n", records[0]["content"][:300])

Created 10 chunks from 5 document(s).

Example chunk:
 Acme Corp - Paid Time Off & Leave Policy
Document 01 | People Operations | Effective January 2025
1. Purpose
This policy describes how paid time off (PTO), sick leave, and other leave is earned and used by
Acme Corp employees. It applies to all regular full-time and part-time employees in the United


## 5. Embed the chunks

Convert each chunk to a vector with the embedding model. We batch the calls for efficiency and read the **dimension** from the first vector (3072 for `text-embedding-3-large`) so the index schema matches automatically.

In [15]:
def embed_texts(texts, batch_size=16):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = aoai.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=batch)
        vectors.extend(d.embedding for d in resp.data)
    return vectors

chunk_vectors = embed_texts([r["content"] for r in records])
EMBED_DIM = len(chunk_vectors[0])
print(f"Embedded {len(chunk_vectors)} chunks. Vector dimension: {EMBED_DIM}")

Embedded 10 chunks. Vector dimension: 3072


## 6. Create the Azure AI Search index

The index defines the schema: a key, the searchable text, a filterable `source`, and the **vector field**. The `VectorSearch` config wires up an HNSW algorithm (fast approximate nearest-neighbor) via a profile that the vector field references.

In [16]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)

# 1) Delete any existing index so we start from a known-good schema.
#    (Azure AI Search forbids changing an existing field, and a leftover index
#    from the portal wizard may use different field names like 'chunk'/'text_vector'.)
try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception as e:
    print("No existing index to delete (ok):", type(e).__name__)

# 2) Define the schema.
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="source", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIM,
        vector_search_profile_name="vprofile",
    ),
]
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

# 3) Create the index fresh (create_index, not create_or_update_index, so there is
#    no "cannot change existing field" conflict).
index_client.create_index(SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search))

# 4) VERIFY: print the fields that now exist. You should see id, content, source, contentVector.
live = index_client.get_index(INDEX_NAME)
print(f"\nIndex '{INDEX_NAME}' created. Fields now present:")
for f in live.fields:
    print("  -", f.name)


Deleted existing index 'ragpipeline-index'.

Index 'ragpipeline-index' created. Fields now present:
  - id
  - content
  - source
  - contentVector


## 7. Upload the chunks + vectors

Push each chunk and its vector into the index. Uploads are batched; `upload_documents` is an upsert, so re-running is safe.

In [17]:
search_client = get_search_client()

docs_to_upload = [
    {"id": r["id"], "content": r["content"], "source": r["source"], "contentVector": v}
    for r, v in zip(records, chunk_vectors)
]

for i in range(0, len(docs_to_upload), 100):
    search_client.upload_documents(documents=docs_to_upload[i:i + 100])

print(f"Uploaded {len(docs_to_upload)} chunks to '{INDEX_NAME}'.")

Uploaded 10 chunks to 'ragpipeline-index'.


## 8. Retrieve

Embed the question, then run a vector (nearest-neighbor) search to pull the top-k most similar chunks. This is the "R" in RAG.

In [19]:
from azure.search.documents.models import VectorizedQuery

def retrieve(query, k=5):
    qvec = embed_texts([query])[0]
    vquery = VectorizedQuery(vector=qvec, k_nearest_neighbors=k, fields="contentVector")
    results = search_client.search(
        search_text=None,            # pure vector search today; hybrid comes on Day 8
        vector_queries=[vquery],
        select=["content", "source"],
    )
    return [
        {"content": r["content"], "source": r["source"], "score": r["@search.score"]}
        for r in results
    ]

# Quick sanity check on retrieval:
for h in retrieve("How many PTO days do employees get?"):
    print(f"[{h['score']:.3f}] {h['source']}: {h['content'][:90]}...")

[0.744] 01_PTO_and_Leave_Policy.pdf: probation (introductory) period. No
PTO is accrued during the first 90 days of employment....
[0.711] 01_PTO_and_Leave_Policy.pdf: Acme Corp - Paid Time Off & Leave Policy
Document 01 | People Operations | Effective Janua...
[0.636] 04_Employee_Benefits_Overview.pdf: Acme Corp - Employee Benefits Overview
Document 04 | People Operations | Effective January...
[0.609] 02_New_Hire_Onboarding_Guide.pdf: ed to your start
date so you do not lose any earned time.
Setting Up for Remote Work
If yo...
[0.605] 04_Employee_Benefits_Overview.pdf: t employees
may use at their discretion.
Wellness
Employees have access to an Employee Ass...


## 9. Generate (the full RAG answer)

Retrieve the relevant chunks, format them as numbered context, and ask the model to answer **only** from them and cite the passage numbers.

In [20]:
def answer_question(question, k=5):
    hits = retrieve(question, k=k)
    context = "\n\n".join(
        f"[{i+1}] (source: {h['source']})\n{h['content']}" for i, h in enumerate(hits)
    )
    system = (
        "You are a question-answering assistant. "
        "Answer ONLY using the numbered context passages below. "
        "Cite the passage numbers you used, e.g. [1], [2]. "
        "If the answer is not in the context, say you don't know based on the documents."
    )
    user = f"Context passages:\n{context}\n\nQuestion: {question}"
    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content, hits

In [21]:
questions = [
    "How many PTO days do full-time employees get?",
    "When do new hires start accruing PTO?",
    "Can I work remotely, and are there any rules?",
    "What is the company's stock price?",  # out of scope -> should decline
]

for q in questions:
    answer, hits = answer_question(q)
    print("Q:", q)
    print("A:", answer)
    print("Sources:", sorted({h["source"] for h in hits}))
    print("-" * 70)

Q: How many PTO days do full-time employees get?
A: Full-time employees at Acme Corp accrue 15 days of paid time off (PTO) per calendar year. After five years of continuous service, the annual PTO allowance increases to 20 days per year [2].
Sources: ['01_PTO_and_Leave_Policy.pdf', '02_New_Hire_Onboarding_Guide.pdf', '03_Remote_Work_Policy.pdf', '04_Employee_Benefits_Overview.pdf']
----------------------------------------------------------------------
Q: When do new hires start accruing PTO?
A: New hires start accruing PTO after completing the 90-day probation (introductory) period. Once the probation period is complete, PTO accrual is backdated to the employee's start date so no earned time is lost [1], [2], [3].
Sources: ['01_PTO_and_Leave_Policy.pdf', '02_New_Hire_Onboarding_Guide.pdf', '04_Employee_Benefits_Overview.pdf']
----------------------------------------------------------------------
Q: Can I work remotely, and are there any rules?
A: Yes, you can work remotely if your role

##  Day 2 checklist

1. Chunks created from your document(s)
2. Chunks embedded and uploaded to an Azure AI Search vector index
3. `retrieve()` returns relevant chunks ranked by score
4. `answer_question()` answers from retrieved context **and cites passages**
5. The out-of-scope question is declined, not hallucinated


### Next up — Day 3: FAQ assistant with robust "I don't know"
Swap in an FAQ dataset, add a **relevance-score threshold** so weak retrievals short-circuit to "not found" *before* calling the LLM, and test systematically with in-scope vs out-of-scope questions.